## MATH 4511 Assignment 1


### Exercise 1

In [2]:
# Exercise 1
T = 5
F = 1200
c = 0.05
m = 2
y = 0.04 # annual effective yield rate of the bond


#a) The current price of the bond
def initial_price(T, F, c, m, y):
    N = int(round(T*m))
    coupons = F*c/m
    P0 = sum(
        coupons/(1+y)**(t/m) for t in range(1, N+1)
    ) + F/(1+y)**(T)
    return P0
print(f'The current price of the bond is ${initial_price(T, F, c, m, y):.2f}')

The current price of the bond is $1256.07


**Modified duration**

$$
D_{Mac}(i) = \frac{\sum_k t_k\, C_k (1+i)^{-t_k}}{\sum_k C_k (1+i)^{-t_k}}, \qquad
D_{Mod}(i) = -\frac{P'(i)}{P(i)} = \frac{D_{Mac}(i)}{1+i}
$$

**Convexity**

$$
C_{Mod}(i) = \frac{P''(i)}{P(i)} = \frac{\sum_k t_k (t_k+1)\, C_k (1+i)^{-t_k-2}}{P(i)}
$$

**Price change for a yield shift $\varepsilon$**

$$
\frac{P(i_0+\varepsilon) - P(i_0)}{P(i_0)} \approx -\varepsilon\, D_{Mod} + \frac{\varepsilon^2}{2}\, C_{Mod}
$$


In [3]:
def d_mac(T, F, c, m, y): # Mac
    coupons = F*c/m
    N = int(round(T*m))
    temp_sum = F*T*(1+y)**(-T)
    for i in range(1, 1+N):
        temp_sum += (i/m)*coupons*(1+y)**(-i/m)
    P0 = initial_price(T, F, c, m, y)
    return (temp_sum / P0)

def d_mod(T, F, c, m, y): # Modified duration
    mac_duration = d_mac(T, F, c, m, y)
    return (mac_duration/(1+y))

def c_mod(T, F, c, m, y):
    coupons = F*c/m
    N = int(round(T*m))
    temp_sum = F*T*(T+1)*(1+y)**(-T-2)
    for i in range(1, N+1):
        temp_sum += coupons*(i/m)*(i/m+1)*(1+y)**(-i/m-2)
    P0 = initial_price(T, F, c, m, y)
    return (temp_sum/P0)

print(f'The modified duration of the bond is {d_mod(T, F, c, m, y):.2f}\nThe modified convexity of the bond is {c_mod(T, F, c, m, y):.2f}')

The modified duration of the bond is 4.33
The modified convexity of the bond is 24.13


In [4]:
#c) Calculate the percentage price in bond price using second order approximation if the annual effective yield of the bond decreases by 0.1%
def second_ord_approx(T, F, c, m, y, eps):
    mod_duration = d_mod(T, F, c, m, y)
    mod_convexity = c_mod(T, F, c, m, y)
    price_change_percentage = -eps*mod_duration + eps**(2)*mod_convexity/2
    return price_change_percentage
print(f'The price change of the bond when the annual effective yield decreases with 0.1% is {second_ord_approx(T, F, c, m, y, eps=-0.001)*100:.2f}% using the second order approximation')

The price change of the bond when the annual effective yield decreases with 0.1% is 0.43% using the second order approximation


### Exercise 2

In [ ]:
import numpy as np
from scipy.optimize import brentq

#a)
spot_rates = { #the spot rates are quoted as annual effective rates
    0.25: 0.021,
    0.50: 0.021,
    0.75: 0.022,
    1.00: 0.024,
    1.25: 0.024,
    1.50: 0.024,
    1.75: 0.026,
    2.00: 0.026,
    2.25: 0.029,
    2.50: 0.029,
    2.75: 0.029,
    3.00: 0.031,
}

T = 3
F = 1000
c = 0.04
m = 2

def B(t, spot_rates): #We assume that t is denoted in years
    discount_rate = 1/(1+spot_rates[t])**(t)
    return discount_rate

# The initial price is the sum of the present value of the cash flows
def initial_price_spot(T, F, c, m, spot_rates): #This is used when the discount rate for the yields are not fixed
    coupons = F*c/m
    N = int(round(T*m))
    temp_sum = F*B(T, spot_rates)
    time = 0
    for t in range(1, N+1):
        temp_sum += coupons*B(t/m, spot_rates)
    return temp_sum

print(f'The intial price of the bond is ${initial_price_spot(T, F, c, m, spot_rates):.2f}')

# The yield of the bond is the discount rate which makes the npv of the bond equal to 0
# This requires solving a ploynomial of degree N. We choose to solve it numerically

def bond_yield(T, F, c, m, spot_rates, tol=1e-4):
    P0 = initial_price_spot(T, F, c, m, spot_rates)
    yield_numerical = brentq(lambda y: initial_price(T, F, c, m, y) - P0, 0.0, 0.5) # The last values are the bounds for the yields we consider
    return yield_numerical
print(f'The annual effective yield of the bond is {bond_yield(T, F, c, m, spot_rates)*100:.4f}%')

def effective_to_nominal(y, m):
    yield_nominal = m*((1+y)**(1/m)-1)
    return yield_nominal
print(f'The annual nominal yield of the bond is {effective_to_nominal(bond_yield(T, F, c, m, spot_rates), m)*100:.4f}%')




The intial price of the bond is $1026.98
The annual effective yield of the bond is 3.0754%
The annual nominal yield of the bond is 3.0522%


In [ ]:
#b) Using the first order approximation we want to calculate the percentage change in bond price when the annual nominal yield is increased with 0.2%
yield_effective = bond_yield(T, F, c, m, spot_rates)
yield_nominal = effective_to_nominal(yield_effective, m)
D_mac = d_mac(T, F, c, m, yield_effective) # The Macaulay duration is the same with annual effective and annual nominal yields
D_mod_nominal = D_mac / (1 + yield_effective/m)
eps = 0.002
print(f'The first order approximation for the percentage change in price is {-eps * D_mod_nominal*100:.4f}%')




The first order approximation for the percentage change in price is -0.5631%


### Exercise 3

In [7]:
# Problem 3 (a): spot rates today (annual effective), key = term in years
spot_rates_0 = {
    0.5: 0.020,
    1.0: 0.020,
    1.5: 0.022,
    2.0: 0.024,
    2.5: 0.025,
    3.0: 0.028,
}

F = 100
T = 3
m = 2
N = T*m
rate_fixed = 0.0025
# The coupon rate is 6-month spot rate + 0.25%
# The price of the bond can be calculated by constructing a portfolio with the same cash flows and then pricing the portfolio
# This note can be decomposed into a pure floating rate note as well as an annuity that pays 0.0025*Face value every 6 months
# The pure floating note has an inital price equal to the face value
def B(t, spot_rates): #We assume that t is denoted in years
    discount_rate = 1/(1+spot_rates[t])**(t)
    return discount_rate
pv_annuity = sum(
    rate_fixed*F*B(t/m, spot_rates_0) for t in range(1, N+1)
)
P0 = F + pv_annuity
print(f'The initial price of the floating rate note is {P0:.2f}')

The initial price of the floating rate note is 101.44


In [ ]:
spot_rates_9m = {
    0.25: 0.018,
    0.75: 0.020,
    1.25: 0.023,
    1.75: 0.026,
    2.25: 0.027,
    2.75: 0.028,
}

# 6-month spot rate at the first coupon date (t = 0.5), annual effective
spot_6m_at_first_coupon = 0.02
coupon1 = ((1 + spot_6m_at_first_coupon)**(0.5) - 1) * F

# Because the the previous spot rate is known the first coupon payment starting from month 9 can be calculated exactly. The coupon is F*S_6(6) where S_6(6) is the 6-spot rate after 6 months 
# We now need to calculate the price of the floating rate note after the first certain coupon has been paid and discount it back, that is calculate the price at month 12 and discount to month 9
# One coupon has already been paid
pv_annuity_9m = sum(
    rate_fixed*F*B(t/m - 3/12, spot_rates_9m) for t in range(1, N+1-1)
) 

price_9m = pv_annuity_9m + (F + coupon1)*B(0.25, spot_rates_9m)
print(f'The price of the floating rate note at t = 9 months is {price_9m:.2f}')


The price of the floating rate not at t = 9 months is 101.76


### Problem 4
Investor has enterd into a 2-year contract on a 5-year coupon bond with face value $1000 and coupon rate 5% payable semi-annually. 
The current annual effective yield of the bond is 5%.
After 1 year the spot rates decrease with 0.5% uniformaly and the annual effective yield becomes 4.5%
The investor wants to "cancel the short position". 

The investor can implement the following strategy: identify all the future cash flows from the contract and construct a portfolio which has the opposite cash flows. The only cash flows is the setup cost of the portfolio as all the future will be 0.
Since the investor has a short position on the forward contract on the coupon paying bond, the future cash flow from the contract is X - S_T where S_T is the price of the bond and X is the agreed upon price. 

To cancel out the future cash flow the investor needs to buy a long position in the coupon paying bond and short a 1-year zero coupon bond with face value X. To cancel out the coupons the investor receives from the long position in the coupon paying bond he needs to enter short positions in zero-coupon bonds with face value and maturity equal to the coupons received from the long position. 

In [40]:
spot_rate_current = {
    0.5: 0.044,
    1: 0.046,
    1.5: 0.048,
    2: 0.051
}

spot_rate_after = {
    0.5: 0.039,
    1: 0.041,
    1.5: 0.043,
    2: 0.046
}
c = 0.05
F = 1000
y = 0.05 # annual effective rate
T = 2
TB = 5
m = 2
N = TB*m
k = T*m

S0 = initial_price(TB, F, c, m, y)
X = (S0 - sum(
    F*c/m*B(t/m, spot_rate_current) for t in range(1, k+1)
)) / B(T, spot_rate_current)

y1 = 0.045
S1 = initial_price(TB-1, F, c, m, y1)

f_1 = S1 - X*B(T-1, spot_rate_after) - sum(
    F*c/m*B(t/m, spot_rate_after) for t in range(1, k+1-2)
)
print(f'The value of the short position in the contract is {-f_1}')

short_pos = X*B(1, spot_rate_after) + sum(
    F*c/m*B(t/m, spot_rate_after) for t in range(1, k+1-2)
) # The money received at t=1 from entering into the short positions
long_pos = -S1 # The cost of entering into a long position at t=1 for the coupon paying bond
print(f'The cost of cancelling out the contract is {short_pos+long_pos}. \nThis is exactly equal to the value of the short position of the contract at time t=1')

The value of the short position in the contract is -7.523930546299873
The cost of cancelling out the contract is -7.52393054629988. 
This is exactly equal to the value of the short position of the contract at time t=1
